# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import os
import getpass
import duckdb
import pandas as pd

# Get Hugging Face token safely
hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    hf_token = getpass.getpass("Enter your Hugging Face READ token: ")

# Connect DuckDB
con = duckdb.connect()

# Register Hugging Face credentials
con.execute("SET VARIABLE hf_token = ?", [hf_token])

con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

# Warehouse paths
REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

print("Connected to FlyRank warehouse")
print("Feature window: February 2026")
print("Label window: March 2026")

Enter your Hugging Face READ token:  ········


Connected to FlyRank warehouse
Feature window: February 2026
Label window: March 2026


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis + time window

**Unit of analysis:** One row represents **one content item for one client**
(`client_hash_id × content_hash_id`).

The source table `fact_content_daily_performance` is at daily
`report_date × client × content` grain. I will aggregate the daily rows
into one client-content row for the feature window.

**Feature window:** February 2026 (`2026-02-01` → `2026-02-28`).

**Decision moment:** 2026-02-28.

**Label window:** March 2026 (`2026-03-01` → `2026-03-31`).

The feature and label windows do not overlap. Features use information
available by the February 28 decision moment; the March outcome is used
only as the label.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [9]:
import duckdb

print("DuckDB version:", duckdb.__version__)

print("\nHTTPFS extension:")
print(
    con.sql("""
        SELECT extension_name, loaded
        FROM duckdb_extensions()
        WHERE extension_name = 'httpfs'
    """).df()
)

print("\nHugging Face secrets:")
print(
    con.sql("""
        SELECT name, type, provider, scope
        FROM duckdb_secrets()
    """).df()
)

DuckDB version: 1.5.4

HTTPFS extension:
  extension_name  loaded
0         httpfs    True

Hugging Face secrets:
  name         type provider    scope
0   hf  huggingface   config  [hf://]


In [10]:
test = con.sql(f"""
    SELECT *
    FROM {DIM_CLIENTS}
    LIMIT 5
""").df()

test

,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,NaT,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,NaT,NaT
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,NaT
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,NaT


### Field classification

**Features — known before the prediction moment**
- `gsc_clicks` — search clicks observed during the feature window
- `gsc_impressions` — search impressions observed during the feature window
- `gsc_avg_position` — average search position during the feature window
- `ga4_sessions` — sessions observed during the feature window
- `ga4_engaged_sessions` — engaged sessions observed during the feature window

**Label / proxy — outcome to predict**
- March 2026 `gsc_clicks` — used as the outcome for ranking content items.

**Context**
- `client_hash_id` — identifies the client/group.
- `content_hash_id` — identifies the content item.
- `report_date` — identifies the daily observation date.

**Excluded**
- `gsc_clicks` and other March 2026 outcome fields are excluded from the feature set because they occur after the February 28 decision moment and would cause future-data leakage.
- Query-level fields from `fact_content_query_90d` are excluded because their fixed 90-day window can overlap the March label window.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
grain_check = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {FEB}
    GROUP BY client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

grain_check

,client_hash_id,content_hash_id,row_count
0,client_3ffa76342f366962,content_46480701a1e637f5,13
1,client_3ffa76342f366962,content_a62547df23fcc492,10
2,client_3ffa76342f366962,content_a0a625c2abc98e6a,10
3,client_3ffa76342f366962,content_036fdef5aeaa8bc0,17
4,client_3ffa76342f366962,content_521a586f8f8f7b49,10


In [12]:
feb_summary = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {FEB}
""").df()

feb_summary

,row_count,min_date,max_date
0,7355108,2026-02-01,2026-02-28


In [14]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows
    FROM {FEB}
""").df()

availability_check

,total_rows,ga4_available_rows
0,7355108,145321


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This dataset has several limitations that affect how the results should be interpreted.

- **Uneven history depth:** Clients do not all have the same amount of GSC or GA4 history, so older observations are not equally available across clients.
- **GA4 availability:** GA4 fields should only be interpreted when `ga4_data_available IS TRUE`. Zero values before the GA4 start date do not necessarily mean zero engagement.
- **Search access differences:** Some clients have limited or no usable search/analytics history, so results may not generalize equally across all clients.
- **Time-window limitation:** The March 2026 label is only an outcome for this contract. It should not be used to construct February features.
- **Query-table limitation:** The 90-day query table has a fixed window that can overlap the label period, so its fields are excluded from this feature set.
  
**Named limitation:** The main limitation is uneven client history and data availability. A client-content row with missing history should not automatically be interpreted as zero activity.

In [15]:
feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_clicks) AS gsc_clicks_feb,
        SUM(gsc_impressions) AS gsc_impressions_feb,
        AVG(gsc_avg_position) AS avg_position_feb,
        SUM(ga4_sessions) AS ga4_sessions_feb,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions_feb

    FROM {FEB}
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

feature_frame.head()

,client_hash_id,content_hash_id,gsc_clicks_feb,gsc_impressions_feb,avg_position_feb,ga4_sessions_feb,ga4_engaged_sessions_feb
0,client_625b6439094e23e4,content_c7bc2833f6f1d043,0.0,0.0,NaN,0.0,0.0
1,client_625b6439094e23e4,content_de0fecfd1b2f01d0,0.0,0.0,NaN,0.0,0.0
2,client_625b6439094e23e4,content_a6b84070e063d1a0,0.0,0.0,NaN,0.0,0.0
3,client_625b6439094e23e4,content_d057ed2d9399f5ed,0.0,0.0,NaN,0.0,0.0
4,client_625b6439094e23e4,content_90fdf47de4aa2506,0.0,0.0,NaN,0.0,0.0


### Feature availability

All five features are calculated from February 2026 data and are available by the February 28, 2026 decision moment.

- `gsc_clicks_feb` — available from GSC observations recorded during February.
- `gsc_impressions_feb` — available from GSC observations recorded during February.
- `avg_position_feb` — available from GSC position observations recorded during February; missing values indicate that a position observation was not available.
- `ga4_sessions_feb` — available from GA4 observations during February when `ga4_data_available IS TRUE`.
- `ga4_engaged_sessions_feb` — available from GA4 observations during February when `ga4_data_available IS TRUE`.

No March 2026 outcome data is used to construct these features.

In [17]:
label_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS gsc_clicks_mar
    FROM {MAR}
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

label_frame.head()

,client_hash_id,content_hash_id,gsc_clicks_mar
0,client_73cda7b4e4f265ea,content_86a300e7753221ca,0.0
1,client_73cda7b4e4f265ea,content_ab290c090bf70554,30.0
2,client_73cda7b4e4f265ea,content_b44471800e34c839,0.0
3,client_73cda7b4e4f265ea,content_2452cf1fb7b5364b,5.0
4,client_73cda7b4e4f265ea,content_ed3e64aa9adc9057,11.0


In [18]:
leakage_test = con.sql(f"""
    WITH feb AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks) AS gsc_clicks_feb
        FROM {FEB}
        GROUP BY client_hash_id, content_hash_id
    ),
    mar AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks) AS gsc_clicks_mar
        FROM {MAR}
        GROUP BY client_hash_id, content_hash_id
    ),
    joined AS (
        SELECT
            feb.client_hash_id,
            feb.content_hash_id,
            feb.gsc_clicks_feb,
            mar.gsc_clicks_mar
        FROM feb
        INNER JOIN mar
            ON feb.client_hash_id = mar.client_hash_id
            AND feb.content_hash_id = mar.content_hash_id
    )
    SELECT
        CORR(gsc_clicks_feb, gsc_clicks_mar) AS honest_score,
        CORR(gsc_clicks_mar, gsc_clicks_mar) AS leaked_score
    FROM joined
""").df()

leakage_test

,honest_score,leaked_score
0,0.775303,1.0


In [20]:
# Deliberately add the March outcome as a leaked feature.
leaky_frame = feature_frame.merge(
    label_frame,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# The March outcome is label-derived and must not remain in the feature set.
leaky_frame = leaky_frame.drop(columns=["gsc_clicks_mar"])

# Keep the original February feature frame as the final honest feature set.
honest_feature_frame = feature_frame.copy()

honest_feature_frame.head()

,client_hash_id,content_hash_id,gsc_clicks_feb,gsc_impressions_feb,avg_position_feb,ga4_sessions_feb,ga4_engaged_sessions_feb
0,client_625b6439094e23e4,content_c7bc2833f6f1d043,0.0,0.0,NaN,0.0,0.0
1,client_625b6439094e23e4,content_de0fecfd1b2f01d0,0.0,0.0,NaN,0.0,0.0
2,client_625b6439094e23e4,content_a6b84070e063d1a0,0.0,0.0,NaN,0.0,0.0
3,client_625b6439094e23e4,content_d057ed2d9399f5ed,0.0,0.0,NaN,0.0,0.0
4,client_625b6439094e23e4,content_90fdf47de4aa2506,0.0,0.0,NaN,0.0,0.0


### Leakage experiment

I deliberately added the March 2026 `gsc_clicks_mar` outcome as a leaked feature to demonstrate label leakage.

The honest February-to-March correlation was **0.775303**, while using the March outcome itself produced a perfect **1.000000** correlation. This artificial score increase occurs because the feature contains the value being predicted.

I removed `gsc_clicks_mar` after the experiment. The final feature frame contains only the five February 2026 features that were available by the February 28, 2026 decision moment.

**Honest score:** 0.775303  
**Leaked score:** 1.000000

The leaked feature is not used in the final analysis.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.